# Product Analytics Case Study: Retention & Conversion Funnel Analysis
## Multi-Category Online Store (REES46 Dataset)

---

## О проекте

Сквозной аналитический кейс на данных реального e-commerce: от знакомства с данными — через SQL-аналитику воронки и когортный анализ — до проверки статистических гипотез и сборки BI-дашборда.

## Используемые данные

[eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) — события пользователей интернет-магазина электроники за октябрь–декабрь 2019 (просмотры, добавления в корзину, покупки).

## Задачи на старте исследования

Прежде чем формулировать какие-либо бизнес-гипотезы, нужно разобраться, что лежит в данных:

1. Оценить объём, структуру и качество данных (пропуски, типы событий, временной охват)
2. Построить базовые метрики воронки (view → cart → purchase)
3. Посмотреть на динамику активности пользователей во времени
4. На основе обнаруженных паттернов — сформулировать конкретную бизнес-проблему для дальнейшего исследования (Ноутбук 2)

## Стек инструментов

- **SQL (DuckDB)** — агрегация, когортный анализ, оконные функции
- **Python (pandas, scipy)** — статистические тесты, обработка данных
- **Yandex DataLens** — финальный дашборд

## Структура данных

Каждая строка датасета — одно событие пользователя (просмотр товара, добавление в корзину или покупка).

| Поле | Описание |
|---|---|
| `event_time` | Время события (UTC) |
| `event_type` | Тип события: `view`, `cart`, `purchase` |
| `product_id` | ID товара |
| `category_id` | ID категории товара |
| `category_code` | Иерархический код категории (напр. `electronics.smartphone`) |
| `brand` | Бренд товара |
| `price` | Цена товара |
| `user_id` | ID пользователя |
| `user_session` | ID сессии (уникален для пары user + визит) |

**Уровень данных:** одно событие = одна строка. Чтобы получить метрики на уровне пользователя или сессии (воронка, retention), потребуется агрегация.

**Период:** октябрь – декабрь 2019 (3 месяца, 3 отдельных CSV-файла).

## Ключевые метрики проекта

- **Conversion rate (view → cart)** — доля просмотров, закончившихся добавлением в корзину
- **Conversion rate (cart → purchase)** — доля корзин, закончившихся покупкой
- **Repeat purchase rate** — доля пользователей с покупкой, совершивших вторую покупку в течение 30/60 дней
- **Time to second purchase** — медианное время между первой и второй покупкой
- **Retention rate по когортам** — доля пользователей определённой когорты (месяц первой покупки), вернувшихся в последующие месяцы

## Загрузка данных

Используем Kaggle API для прямой загрузки датасета в среду Colab.

In [3]:
!pip install -q kaggle

import json, os
from google.colab import userdata

kaggle_username = userdata.get('kaggle_username')
kaggle_key = userdata.get('kaggle_key')

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [4]:
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store -f 2019-Oct.csv -p /content/data
!unzip -o /content/data/2019-Oct.csv.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
100% 1.61G/1.61G [00:14<00:00, 118MB/s]

Archive:  /content/data/2019-Oct.csv.zip
  inflating: /content/data/2019-Oct.csv  


In [5]:
import os
path = "/content/data/2019-Oct.csv"
print(f"{os.path.getsize(path) / (1024*1024):.1f} MB")

5406.0 MB


In [7]:
import duckdb

con = duckdb.connect("ecommerce.duckdb")

con.execute("""
    COPY (
        SELECT *
        FROM read_csv_auto('/content/data/2019-Oct.csv')
        WHERE abs(hash(user_id)) % 100 < 15
    ) TO '/content/data/oct_sample.csv' (HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
con.execute(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM read_csv_auto('/content/data/oct_sample.csv')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,users
0,6367907,453083


In [9]:
con.execute("""
    CREATE OR REPLACE TABLE events_oct AS
    SELECT * FROM read_csv_auto('/content/data/oct_sample.csv')
""")

con.execute("DESCRIBE events_oct").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,null,key,default,extra
0,event_time,TIMESTAMP,YES,None,None,None
1,event_type,VARCHAR,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,category_id,BIGINT,YES,None,None,None
4,category_code,VARCHAR,YES,None,None,None
5,brand,VARCHAR,YES,None,None,None
6,price,DOUBLE,YES,None,None,None
7,user_id,BIGINT,YES,None,None,None
8,user_session,VARCHAR,YES,None,None,None


In [10]:
con.execute("SELECT * FROM events_oct LIMIT 5").df()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:11,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.64,530282093,50a293fb-5940-41b2-baf3-17af0e812101
1,2019-10-01 00:00:17,view,23100006,2053013561638126333,None,None,357.79,513642368,17566c27-0a8f-4506-9f30-c6a2ccbf583b
2,2019-10-01 00:00:23,view,6200260,2053013552293216471,appliances.environment.air_heater,midea,47.62,538645907,7d9a8784-7b6c-426e-9924-9f688812fd71
3,2019-10-01 00:00:25,view,27500014,2053013554692358509,None,redmond,37.98,555217733,74d40a28-41f9-4325-bbae-b179bd2c0a38
4,2019-10-01 00:00:28,view,26200591,2053013563693335403,None,None,203.35,548449430,99617d1c-1b5a-42f8-99f1-42ad83a6155f


## Первичный обзор данных

Прежде чем переходить к построению воронки и когортного анализа, посмотрим на общую картину: распределение типов событий, долю пропущенных значений в ключевых полях и временной диапазон данных.

In [11]:
# Распределение событий по типам
print("Event types:")
print(con.execute("""
    SELECT event_type, COUNT(*) AS cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM events_oct
    GROUP BY event_type
    ORDER BY cnt DESC
""").df())

# Временной диапазон
print("\nDate range:")
print(con.execute("""
    SELECT MIN(event_time) AS start_date, MAX(event_time) AS end_date
    FROM events_oct
""").df())

# Пропуски в ключевых полях
print("\nMissing values:")
print(con.execute("""
    SELECT
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS missing_category,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS missing_brand,
        COUNT(*) AS total
    FROM events_oct
""").df())

Event types:
  event_type      cnt    pct
0       view  6119511  96.10
1       cart   137439   2.16
2   purchase   110957   1.74

Date range:
           start_date            end_date
0 2019-10-01 00:00:11 2019-10-31 23:59:57

Missing values:
   missing_category  missing_brand    total
0         2024249.0       913081.0  6367907


### Распределение событий на полном исходном датасете для проверки репрезентативности выборки


In [6]:
print("Распределение типов событий на ПОЛНОМ датасете (100%):")

con.execute("""
    SELECT
        event_type,
        COUNT(*) AS event_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM read_csv_auto('/content/data/2019-Oct.csv')
    GROUP BY event_type
    ORDER BY pct DESC
""").df()

Распределение типов событий на ПОЛНОМ датасете (100%):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,event_type,event_count,pct
0,view,40779399,96.07
1,cart,926516,2.18
2,purchase,742849,1.75


## Выводы по первичному обзору

- Датасет за октябрь 2019 (15% выборка пользователей) содержит **6.37 млн событий** от **453 тыс. пользователей**
- Воронка событий: **96.1% view → 2.16% cart → 1.74% purchase** — типичное распределение для e-commerce, где просмотр товара значительно превышает количество покупок
> **Важное методологическое уточнение:** Данные цифры
показывают распределение типов событий в общем объёме логов (долю строк в датасете), а не конверсию. Реальную user-level конверсию (по уникальным пользователям) мы посчитаем далее в Ноутбуке 2, так как специфика сырых данных e-commerce требует агрегации сессий.
- Данные покрывают полный месяц (1–31 октября 2019) без пропусков по датам
- В полях `category_code` (32% пропусков) и `brand` (14% пропусков) есть значимая доля отсутствующих значений — это нужно учитывать при анализе на уровне категорий и брендов, но не влияет на анализ воронки и retention на уровне пользователей/сессий

**Далее:** переходим к построению воронки конверсии и когортного retention-анализа в SQL (Ноутбук 2).

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
!mkdir -p /content/drive/MyDrive/ecommerce_project/data
!cp /content/data/oct_sample.csv /content/drive/MyDrive/ecommerce_project/data/